# 03 · RQ3 breakout

**Question.** What separates trends that spread to other platforms from those that stay on one? The answer becomes the features of the hours horizon's forecast ("will this reach 3 platforms within 6 hours?").

**Data.** The stored lists (`trend_items`) through RQ6's matching (`topnews.echo`). Once topic snapshots exist, repeat this on them, with `news_count` and each topic's first hours as features.

**Method.** Breakouts to 3 or more platforms are too rare so far (RQ6), so the outcome is *spread*: matched on at least one other list at 0.70 (stricter than RQ6's 0.60, because a wrong match makes a wrong label). Features are what's known at a trend's first sighting: platform, entry rank, its metric against its list's median, hour of day (US Eastern) and title length. Only trends whose entry was seen, and that had at least 6 hours of data after it, count: a trend seen in the last hours hasn't failed to spread, it hasn't had time. Rates come with 95% Wilson intervals, which stay honest at small counts.

**Needs** `npm ci` at the repo root, as notebook 06 does. Run top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
import pandas as pd

from topnews import breakout, db, echo

pd.set_option("display.max_colwidth", 50)
engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
items = db.trend_items(engine, days=28)  # YouTube left out

def population(subset, name):
    cp, _ = echo.analyze(subset, name=name)
    table = breakout.observable(breakout.features(breakout.label(cp), subset), subset)
    table = breakout.buckets(table)
    print(f"{len(table)} trends with a seen entry and 6 hours to spread; {int(table.spread.sum())} spread, {int(table.breakout.sum())} reached 3+ platforms")
    return table

def rate_tables(table):
    for by in ["source_id", "entry", "metric", "part_of_day", "length", "hashtag"]:
        display(breakout.rates(table, by).round(3))

## A · The phase-1 lists, collected all day

Bluesky, Google Trends, Hacker News, Mastodon and Twitch, matched only against each other, so every trend had the same lists to spread to.

In [ ]:
a = population(items[items.source_id.isin(["bluesky", "google_trends", "hacker_news", "mastodon", "twitch"])], "rq3_a")
rate_tables(a)

## B · Every list, from when all of them are collected

RQ6 found most sharing runs through X, Google and Instagram, which the phase-1 lists can't show. This cut starts when the last source joined (2026-09-30 22:08 UTC), so it fills as the days go by.

In [ ]:
b = population(echo.since_all_present(items), "rq3_b")
rate_tables(b)

## The trends that spread, for reading

Check each match by eye: a wrong one is a wrong label.

In [ ]:
columns = ["source_id", "title", "entry_rank", "metric_vs_median", "hour_local", "platforms", "best_platform", "best_title", "best_sim"]
pd.concat({"A": a[a.spread][columns], "B": b[b.spread][columns]}).round(2)

## A first model, once there's enough

A logistic regression of spread on platform, entry rank, metric and hour needs about 30 trends that spread before its coefficients mean anything. Until then this only says how many there are.

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

for label, table in [("A", a), ("B", b)]:
    positives = int(table.spread.sum())
    if positives < 30:
        print(f"{label}: {positives} trends spread; the model waits for 30.")
        continue
    data = table.assign(spread=table.spread.astype(int), log_metric=np.log(table.metric_vs_median.clip(lower=0.01)))
    model = smf.logit("spread ~ C(source_id) + entry_rank + C(part_of_day)", data=data.dropna(subset=["entry_rank"])).fit(disp=False)
    print(label)
    display(model.summary2().tables[1].round(3))